# 01 · Run one complete model day

Choose one `MODEL_ID`, review the derived status, then explicitly enable the
expensive stage. Rerunning all cells resumes only compatible unfinished work.
Search rungs, final checkpoints, evaluation, profiling, and reports are
discovered automatically from persistent storage.


In [ ]:
MODEL_ID = "rtdetrv2_l"

RUN_MODE = "auto"
RUN_LR_RANGE_TEST = True
RUN_BOUNDARY_EXTENSION = False

START_EXPENSIVE_STAGE = False
ALLOW_OVER_BUDGET_RUN = False


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
REPO_PATH = Path("/content/aerial-object-detection-benchmark") if IN_COLAB else Path.cwd()
if IN_COLAB:
    if not (REPO_PATH / ".git").is_dir():
        subprocess.run(
            ["git", "clone", "--branch", "main", "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git", str(REPO_PATH)],
            check=True,
        )
    elif subprocess.check_output(["git", "-C", str(REPO_PATH), "status", "--porcelain"], text=True).strip():
        raise RuntimeError("Repository has local changes; refusing to update it.")
    else:
        subprocess.run(["git", "-C", str(REPO_PATH), "pull", "--ff-only", "origin", "main"], check=True)
sys.path.insert(0, str(REPO_PATH))
DRIVE_ROOT = (
    Path("/content/drive/MyDrive/visdrone_architecture_benchmark")
    if IN_COLAB
    else Path(os.environ.get("VISDRONE_DRIVE_ROOT", REPO_PATH / "local_artifacts"))
)
SMOKE_TEST = os.environ.get("SMOKE_TEST", "").lower() in {"1", "true", "yes"}


In [ ]:
from src.workflows.model_day import ModelDayOptions, inspect_model_day
state = inspect_model_day(DRIVE_ROOT, MODEL_ID, REPO_PATH)
print(json.dumps(state, indent=2, default=str))
if state["stage"] == "DATA":
    raise RuntimeError(
        "Dataset setup is incomplete. Run 00_prepare_visdrone.ipynb, then rerun this notebook."
    )


In [ ]:
from src.workflows.model_day import run_model_day
result = run_model_day(
    REPO_PATH,
    DRIVE_ROOT,
    ModelDayOptions(
        model_id=MODEL_ID,
        run_mode=RUN_MODE,
        run_lr_range_test=RUN_LR_RANGE_TEST,
        run_boundary_extension=RUN_BOUNDARY_EXTENSION,
        start_expensive_stage=START_EXPENSIVE_STAGE,
        allow_over_budget_run=ALLOW_OVER_BUDGET_RUN,
        smoke_test=SMOKE_TEST,
    ),
)
print(json.dumps(result, indent=2, default=str))


In [ ]:
if result["stage"] == "COMPLETE":
    evaluation = json.loads(Path(result["evaluation_paths"][0]).read_text())
    print("MODEL DAY COMPLETE")
    print()
    print(f"Model: {MODEL_ID}")
    print(f"Selected LR: {result['selected_lr']}")
    print(f"Final run ID: {result['final_run_id']}")
    print(f"Best checkpoint: {result['checkpoint_path']}")
    print(f"mAP50-95: {evaluation.get('mAP')}")
    print(f"APtiny: {evaluation.get('APtiny')}")
    print(f"Report: {result['report_path']}")
    print(f"Recommended bundle: {result['recommended_bundle']}")
    print("Next notebook: 02_publish_results.ipynb")
else:
    print(result.get("message", f"Next stage: {result['stage']}"))
